### 1. Load Benchmark Problems

In [ ]:
sampling_for_lite = False # True for MineCEraft Lite
# random_seed: single source of truth from builder.json (benchmark sampling + LLM seed when supported)
from pathlib import Path
import json
_builder_path = Path.cwd().parent / "builder.json"
_builder_cfg = json.loads(_builder_path.read_text(encoding="utf-8")) if _builder_path.exists() else {}
sampling_rand_seed = _builder_cfg.get("random_seed", 42)

In [2]:
from pathlib import Path
import json
import random

# Load JSON files from benchmarks folder only (not archive/; archive = excluded from evaluation)
# Schema: prompts = [[turn1, turn2, ...], ...], checks = [[eval_turn1], [eval_turn2], ...]
BENCHMARKS_DIR = Path.cwd() / "benchmarks"
json_files = sorted(BENCHMARKS_DIR.glob("*.json"))
mapping = []
for json_file in json_files:
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

# Build runs: each run = (prompt_sequence, checks_per_turn, _comment)
runs = []
if sampling_for_lite:
    rng = random.Random(sampling_rand_seed)
    for json_file in json_files:
        file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
        file_runs = []
        for item in file_mapping:
            for prompt_sequence in item["prompts"]:
                file_runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))
        if file_runs:
            runs.append(rng.choice(file_runs))
else:
    for item in mapping:
        for prompt_sequence in item["prompts"]:
            runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))

print(f"[PY] Benchmarks dir: {BENCHMARKS_DIR.resolve()} (cwd: {Path.cwd().resolve()})")
if len(runs) == 0:
    print("[PY] ⚠️ No runs. Put at least one .json in benchmarks/ (not in archive/), or run notebook from the folder that contains benchmarks/.")
print("Total problem #:", len(runs))
for i, (prompts, _, _) in enumerate(runs):
    print(i, prompts)

[PY] Benchmarks dir: D:\git\mineCEraft\mineCEraft\benchmarks (cwd: D:\git\mineCEraft\mineCEraft)
Total problem #: 25
0 ['Lay the foundation for a 7x10 block rectangular building. Use stone blocks and make it two blocks deep.']
1 ['Install a 7×7 pile cap, 1 block thick, with four 1×1 drilled piles, each 5 blocks deep, with a 1-block span. Use stone blocks.']
2 ['Erect a stone wall that is 10 blocks wide, 2 blocks thick, and 7 blocks high.']
3 ['Construct a 5 × 5 × 1 flat roof (one block thick) supported by exactly 5 columns with a 1 × 1 cross-section and a height of 3 blocks. Use stone for the columns and oak planks for the roof.']
4 ['Create a 10 × 10 flat roof supported by exactly five 2 x 2 x 2 columns. Arrange the columns so that the roof is as structurally stable as possible. Use stone for the columns and oak planks for the roof.']
5 ['Create a 10 × 10 × 2 flat roof (two block thick) supported by exactly five 2 x 2 x 2 columns. Arrange the columns so that the roof is as structurall

### 2. Build phase (construction)

This cell runs the construction (builder agent) only and writes an `eval_*_raw.json` file that contains, for each prompt, its checks and cumulative coordinates. Run the "Load Benchmark Problems" cell above first so that `runs` is defined.

In [3]:
from pathlib import Path
from build_eval_raw import run_build_and_save_eval_raw

# Build phase:
# - uses `runs` constructed in the "Load Benchmark Problems" cell
# - talks to the builder agent
# - writes a single eval_{model_safe}_{ts}_raw.json file under eval_results/

try:
    runs  # type: ignore[name-defined]
except NameError as exc:
    raise RuntimeError("`runs` is not defined. Run the 'Load Benchmark Problems' cell above first.") from exc

eval_raw_path = run_build_and_save_eval_raw(runs)
# For downstream evaluation we treat eval_raw as a list of files.
eval_raw_files = [eval_raw_path]

print(f"[PY] eval_raw file for evaluation: {eval_raw_path}")

[PY] Using builder model: gemini-3-pro-preview (safe='gemini-3-pro-preview')
[PY] Total turns to send: 30
[PY] Intermediate eval_raw file: eval_results\eval_gemini-3-pro-preview_20260309_171831_raw.json
[PY] Builder agent started (PID=40036)
[PY] Agent log: eval_results\eval_gemini-3-pro-preview_20260309_171831_agent.log
ℹ️ Using agent name: builder
🧹 Cleared all files under D:\git\mineCEraft\bots\builder\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=7vxqBtmiPuDJ4x1-AAAD)

➡️ Sending to builder (run 1, turn 1/1): "Lay the foundation for a 7x10 block rectangular building. Use stone blocks and make it two blocks deep."
⏳ Waiting for completion keyword (timeout 10 min)...
📨 [builder] Laying the 7x10 stone foundation, two blocks deep. !newAction("Build a 7x10 rectangular foundation using stone blocks. Dig a 7x10 hole that is 2 blocks deep starting from the current position, then fill the entire 7x10x2 volume with stone blocks.")
📨 [builder] I've successfully bui

### 3. Evaluation phase (eval_raw → log/csv)

This cell reads one or more `eval_*_raw.json` files and produces `eval_{model_safe}_{ts}.log` and `eval_{model_safe}_{ts}.csv`. It also populates `coords_by_problem` for the visualization cell.

In [4]:
from eval_from_raw import evaluate_from_raw

## Optional: evaluate a custom list of eval_*_raw.json files instead of the default one.
## Example:
# eval_raw_files = [
#     "eval_results/eval_meta-llama-llama-4-scout-17b-16e-instruct_20260305_122722_raw.json",
# ]

# Default: evaluate the eval_raw file produced in the build phase above.
coords_by_problem, log_path, csv_path = evaluate_from_raw(eval_raw_files)

print(f"[PY] Log path: {log_path}")
print(f"[PY] CSV path: {csv_path}")


[PY] === Evaluation Result ===
[PY] Run #5, Turn #1/1: Create a 10 × 10 flat roof supported by exactly five 2 x 2 x 2 columns. Arrange the columns so that the roof is as structurally stable as possible. Use stone for the columns and oak planks for the roof.
[PY] Score: 5.6236085000683484 / 6 (coords=140)
[PY] Category scores:
  - dependency: 1.0 / 1
  - material: 1.0 / 1
  - physical_plausibility: 1.0 / 1
  - shape: 2.0 / 2
  - structural_stability: 0.623608500068348 / 1
  · PASS | eval_code.physical_plausibility.is_ground_connected({})
  · 0.62 | eval_code.structural_stability.stress_score({'von_mises_stress': 62000000.0})
  · PASS | eval_code.material.is_block_cnt_geq({'min_count': 140})
  · PASS | eval_code.shape.cluster_count_at_y_is({'y': 0, 'expected_count': 5})
  · PASS | eval_code.shape.cluster_count_at_y_is({'y': 1, 'expected_count': 5})
  · PASS | eval_code.dependency.material_sequence_order({'material_sequence': ['stone', 'oak_planks']})



[PY] === Evaluation Result ===
[PY] Run #6, Turn #1/1: Create a 10 × 10 × 2 flat roof (two block thick) supported by exactly five 2 x 2 x 2 columns. Arrange the columns so that the roof is as structurally stable as possible. Use white_concrete for the columns, iron for the first layer of the roof, and gray_concrete for the second.
[PY] Score: 5.7569226830382085 / 6 (coords=240)
[PY] Category scores:
  - dependency: 1.0 / 1
  - material: 1.0 / 1
  - physical_plausibility: 1.0 / 1
  - shape: 2.0 / 2
  - structural_stability: 0.7569226830382084 / 1
  · PASS | eval_code.physical_plausibility.is_ground_connected({})
  · 0.76 | eval_code.structural_stability.stress_score({'von_mises_stress': 8943})
  · PASS | eval_code.material.is_block_cnt_geq({'min_count': 240})
  · PASS | eval_code.shape.cluster_count_at_y_is({'y': 0, 'expected_count': 5})
  · PASS | eval_code.shape.cluster_count_at_y_is({'y': 1, 'expected_count': 5})
  · PASS | eval_code.dependency.material_sequence_order({'material_seq